In [25]:
pip install optuna


In [26]:
# ─────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer

import xgboost as xgb
import lightgbm as lgb

from imblearn.over_sampling import SMOTE
import optuna
import shap

optuna.logging.set_verbosity(optuna.logging.WARNING)





In [27]:
# ─────────────────────────────────────────────
# 2. DATA LOADING
# ─────────────────────────────────────────────
def load_data():
    """
    Load the Cleveland Heart Disease dataset directly from UCI repository.
    You can also load from a local CSV: pd.read_csv('heart.csv')
    """
    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "heart-disease/processed.cleveland.data"
    )
    column_names = [
        "age", "sex", "cp", "trestbps", "chol", "fbs",
        "restecg", "thalach", "exang", "oldpeak",
        "slope", "ca", "thal", "target"
    ]
    df = pd.read_csv(url, names=column_names, na_values="?")
    # Binary target: 0 = no disease, 1 = disease
    df["target"] = (df["target"] > 0).astype(int)
    print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Class distribution:\n{df['target'].value_counts()}\n")
    return df


In [28]:
# ─────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
def run_eda(df):
    print("=" * 50)
    print("EDA SUMMARY")
    print("=" * 50)
    print(df.describe())
    print(f"\nMissing values:\n{df.isnull().sum()}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Target distribution
    df["target"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#3B8BD4", "#E8593C"])
    axes[0, 0].set_title("Target Distribution")
    axes[0, 0].set_xticklabels(["No Disease", "Disease"], rotation=0)

    # Age distribution by target
    df.groupby("target")["age"].plot(kind="hist", alpha=0.6, bins=20, ax=axes[0, 1])
    axes[0, 1].set_title("Age Distribution by Target")
    axes[0, 1].legend(["No Disease", "Disease"])

    # Correlation heatmap
    sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
                ax=axes[1, 0], annot_kws={"size": 7})
    axes[1, 0].set_title("Feature Correlation Heatmap")

    # Max heart rate vs Age
    axes[1, 1].scatter(
        df[df["target"] == 0]["age"], df[df["target"] == 0]["thalach"],
        alpha=0.5, color="#3B8BD4", label="No Disease"
    )
    axes[1, 1].scatter(
        df[df["target"] == 1]["age"], df[df["target"] == 1]["thalach"],
        alpha=0.5, color="#E8593C", label="Disease"
    )
    axes[1, 1].set_title("Age vs Max Heart Rate")
    axes[1, 1].set_xlabel("Age")
    axes[1, 1].set_ylabel("Max Heart Rate (thalach)")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig("eda_plots.png", dpi=120, bbox_inches="tight")
    plt.close()
    print("\nEDA plots saved to eda_plots.png\n")


In [29]:

# ─────────────────────────────────────────────
# 4. PREPROCESSING & FEATURE ENGINEERING
# ─────────────────────────────────────────────
def preprocess(df):
    df = df.copy()

    # Impute missing values
    num_cols = ["ca", "thal"]
    for col in num_cols:
        df[col].fillna(df[col].median(), inplace=True)

    # Feature engineering: interaction features
    df["age_thalach"]   = df["age"] * df["thalach"]          # age × max heart rate
    df["bp_chol_ratio"] = df["trestbps"] / (df["chol"] + 1)  # blood pressure / cholesterol
    df["age_group"]     = pd.cut(df["age"], bins=[0, 40, 55, 70, 100],
                                  labels=[0, 1, 2, 3]).astype(int)

    # One-hot encode categorical columns
    cat_cols = ["cp", "restecg", "slope", "thal"]
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    X = df.drop("target", axis=1)
    y = df["target"]

    print(f"Feature matrix shape after engineering: {X.shape}")
    return X, y


In [30]:
# ─────────────────────────────────────────────
# 5. TRAIN / TEST SPLIT + SMOTE
# ─────────────────────────────────────────────
def split_and_balance(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Scale features
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # SMOTE to handle class imbalance on training set only
    sm = SMOTE(random_state=42)
    X_train_bal, y_train_bal = sm.fit_resample(X_train_sc, y_train)

    print(f"Training set after SMOTE: {X_train_bal.shape[0]} samples")
    print(f"Test set: {X_test_sc.shape[0]} samples\n")
    return X_train_bal, X_test_sc, y_train_bal, y_test, scaler


In [31]:
# ─────────────────────────────────────────────
# 6. BASELINE MODELS (cross-validated)
# ─────────────────────────────────────────────
def train_baseline_models(X_train, y_train):
    print("=" * 50)
    print("BASELINE MODEL COMPARISON (10-fold CV)")
    print("=" * 50)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
        "XGBoost":             xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss",
                                                  random_state=42),
        "LightGBM":            lgb.LGBMClassifier(random_state=42, verbose=-1),
        "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
        "SVM":                 SVC(probability=True, random_state=42),
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    results = {}
    for name, model in models.items():
        auc   = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
        f1    = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1").mean()
        acc   = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean()
        results[name] = {"AUC-ROC": auc, "F1": f1, "Accuracy": acc}
        print(f"{name:<25}  AUC={auc:.4f}  F1={f1:.4f}  Acc={acc:.4f}")

    print()
    return models, results


In [32]:
# ─────────────────────────────────────────────
# 7. HYPERPARAMETER TUNING (Optuna)
# ─────────────────────────────────────────────
def tune_xgboost(X_train, y_train, n_trials=50):
    print("=" * 50)
    print(f"TUNING XGBoost ({n_trials} Optuna trials)...")
    print("=" * 50)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
            "max_depth":         trial.suggest_int("max_depth", 3, 10),
            "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 1e-5, 10, log=True),
            "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
            "use_label_encoder": False,
            "eval_metric":       "logloss",
            "random_state":      42,
        }
        model = xgb.XGBClassifier(**params)
        return cross_val_score(model, X_train, y_train,
                               cv=cv, scoring="roc_auc").mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params
    best_params.update({"use_label_encoder": False, "eval_metric": "logloss",
                        "random_state": 42})
    print(f"Best XGBoost AUC (CV): {study.best_value:.4f}")
    print(f"Best params: {best_params}\n")

    best_xgb = xgb.XGBClassifier(**best_params)
    return best_xgb


def tune_random_forest(X_train, y_train, n_trials=40):
    print("TUNING Random Forest...")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def objective(trial):
        params = {
            "n_estimators":       trial.suggest_int("n_estimators", 100, 600),
            "max_depth":          trial.suggest_int("max_depth", 3, 20),
            "min_samples_split":  trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf":   trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features":       trial.suggest_categorical("max_features",
                                                             ["sqrt", "log2", None]),
            "random_state": 42,
        }
        model = RandomForestClassifier(**params)
        return cross_val_score(model, X_train, y_train,
                               cv=cv, scoring="roc_auc").mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params = {**study.best_params, "random_state": 42}
    print(f"Best RF AUC (CV): {study.best_value:.4f}\n")

    return RandomForestClassifier(**best_params)


In [33]:
# ─────────────────────────────────────────────
# 8. ENSEMBLE: STACKING
# ─────────────────────────────────────────────
def build_ensemble(best_xgb, best_rf):
    """
    Stacking ensemble: XGBoost + RF + LightGBM as base learners,
    Logistic Regression as meta-learner.
    """
    base_estimators = [
        ("xgb",  best_xgb),
        ("rf",   best_rf),
        ("lgbm", lgb.LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)),
    ]
    meta_learner = LogisticRegression(max_iter=1000, random_state=42)

    stacking_clf = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta_learner,
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1,
    )
    return stacking_clf


In [34]:
# ─────────────────────────────────────────────
# 9. EVALUATION
# ─────────────────────────────────────────────
def evaluate_model(model, X_train, y_train, X_test, y_test, name="Model"):
    model.fit(X_train, y_train)

    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]

    acc   = accuracy_score(y_test, y_pred)
    auc   = roc_auc_score(y_test, y_prob)
    f1    = f1_score(y_test, y_pred)

    print(f"\n{'=' * 50}")
    print(f"FINAL RESULTS: {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

    # Confusion matrix
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=["No Disease", "Disease"],
                yticklabels=["No Disease", "Disease"])
    axes[0].set_title(f"{name} - Confusion Matrix")
    axes[0].set_ylabel("Actual")
    axes[0].set_xlabel("Predicted")

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, color="#3B8BD4", lw=2,
                 label=f"ROC Curve (AUC = {auc:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", lw=1)
    axes[1].set_title(f"{name} - ROC Curve")
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"results_{name.lower().replace(' ', '_')}.png",
                dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Results plot saved.\n")

    return {"accuracy": acc, "auc": auc, "f1": f1}


In [35]:
# ─────────────────────────────────────────────
# 10. SHAP EXPLAINABILITY
# ─────────────────────────────────────────────
def explain_with_shap(model, X_train, feature_names):
    print("Generating SHAP feature importance...")
    try:
        explainer    = shap.TreeExplainer(model)
        shap_values  = explainer.shap_values(X_train)

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_train,
                          feature_names=feature_names,
                          plot_type="bar", show=False)
        plt.tight_layout()
        plt.savefig("shap_importance.png", dpi=120, bbox_inches="tight")
        plt.close()
        print("SHAP importance plot saved to shap_importance.png\n")
    except Exception as e:
        print(f"SHAP skipped: {e}\n")


In [36]:
# ─────────────────────────────────────────────
# 11. PREDICT ON NEW PATIENT (inference demo)
# ─────────────────────────────────────────────
def predict_new_patient(model, scaler, feature_names):
    """
    Example: predict for a single new patient.
    Fill in real values before using in production.
    """
    # Sample patient data (must match your feature columns)
    sample = {
        "age": 55, "sex": 1, "trestbps": 140, "chol": 250,
        "fbs": 0, "thalach": 145, "exang": 1, "oldpeak": 2.5,
        "ca": 0,
        # engineered features
        "age_thalach": 55 * 145, "bp_chol_ratio": 140 / 251, "age_group": 2,
    }
    # Fill missing one-hot columns with 0
    row = pd.DataFrame([{col: sample.get(col, 0) for col in feature_names}])
    row_sc = scaler.transform(row)

    prob = model.predict_proba(row_sc)[0][1]
    pred = "DISEASE DETECTED" if prob >= 0.5 else "No disease"
    print(f"\nNew patient prediction: {pred}  (probability = {prob:.3f})")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    # 1. Load
    df = load_data()

    # 2. EDA
    run_eda(df)

    # 3. Preprocess & engineer features
    X, y = preprocess(df)
    feature_names = list(X.columns)

    # 4. Split + balance
    X_train, X_test, y_train, y_test, scaler = split_and_balance(X, y)

    # 5. Baseline comparison
    models, baseline_results = train_baseline_models(X_train, y_train)

    # 6. Tune top models
    best_xgb = tune_xgboost(X_train, y_train, n_trials=50)
    best_rf  = tune_random_forest(X_train, y_train, n_trials=40)

    # 7. Evaluate tuned XGBoost
    evaluate_model(best_xgb, X_train, y_train, X_test, y_test, "Tuned XGBoost")

    # 8. Stacking ensemble
    ensemble = build_ensemble(best_xgb, best_rf)
    evaluate_model(ensemble, X_train, y_train, X_test, y_test, "Stacking Ensemble")

    # 9. SHAP explainability (on XGBoost)
    best_xgb.fit(X_train, y_train)
    explain_with_shap(best_xgb, X_train, feature_names)

    # 10. Inference demo
    predict_new_patient(best_xgb, scaler, feature_names)

    print("\nAll done! Check the saved PNG plots for visuals.")


if __name__ == "__main__":
    main()

Dataset loaded: 303 rows, 14 columns
Class distribution:
target
0    164
1    139
Name: count, dtype: int64

EDA SUMMARY
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.438944    0.679868    3.158416  131.689769  246.693069    0.148515   
std      9.038662    0.467299    0.960126   17.599748   51.776918    0.356198   
min     29.000000    0.000000    1.000000   94.000000  126.000000    0.000000   
25%     48.000000    0.000000    3.000000  120.000000  211.000000    0.000000   
50%     56.000000    1.000000    3.000000  130.000000  241.000000    0.000000   
75%     61.000000    1.000000    4.000000  140.000000  275.000000    0.000000   
max     77.000000    1.000000    4.000000  200.000000  564.000000    1.000000   

          restecg     thalach       exang     oldpeak       slope          ca  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  2